<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/MiniMax-H3_ComfyUI_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


#@title <center>MiniMax-H3 (ComfyUI runtime) — L4 + Colab Pro+ (22 GB VRAM, 53 GB host RAM)</center>

# MiniMax-H3 via ComfyUI subprocess + Comfy-Org quantized weights
#
# **Why ComfyUI instead of diffusers?**
#
# This is the second attempt at MiniMax-H3 in this repo. The first attempt (the diffusers-based
# notebook `MiniMax-H3_Colab.ipynb`) hit a hard wall: the released weights are 33B dense
# transformer + 32B Qwen3-VL text encoder, and the initial public open implementation only
# supports **full attention**. There is no sparse-attention inference in the released open
# runtime, and full BF16 plus the text encoder plus the VAEs plus activation buffers cannot
# coexist inside 22 GB of VRAM.
#
# Stock diffusers does not have a knob for the failure mode that actually OOM-kills the run
# here: host RAM being pinned to 90% of system memory by the framework's staging policy.
# ComfyUI does — the `--disable-pinned-memory` launch flag is the difference between an OOM-kill
# on 32 GB host RAM and a 15-second video completing in 23 minutes on the same card. (See
# `tonyd2wild/minimax-h3-local` for the verified result on a 3090 + 31 GB RAM.) We reuse the
# same recipe for our L4 + 53 GB envelope.
#
# **What this notebook runs**
#
# - ComfyUI v0.30.1+ as a subprocess on `127.0.0.1:8188`, launched with
#   `--disable-pinned-memory --fp16-intermediates --listen 127.0.0.1 --port 8188`.
# - The `Comfy-Org/MiniMax-H3` `minimax_h3_fl2va_pruned_int8_convrot` diffusion transformer
#   (19.5 GB on disk) + the `qwen3vl_32b_minimax_h3_nvfp4_awq` text encoder (15.7 GB).
# - A pure-Python driver that builds a t2v / fl2v / r2v API-format workflow JSON, POSTs to
#   `/prompt`, polls `/history`, downloads the result.
#
# **Hardware target**
#
# - GPU: NVIDIA L4 22 GB (also A100-40GB, RTX 3090/4090 24 GB).
# - Host RAM: 53 GB on Colab Pro+. Total weight disk footprint is ~40.8 GB (int8_convrot + NVFP4
#   text encoder + fp16 video VAE + fp32 audio VAE), leaving ~12 GB host RAM free for the
#   framework.
# - Audio is supported end-to-end (the model generates its own stereo soundtrack via the
#   included fp32 audio VAE).
#
# **L4 tuning for first run**
#
# - Resolution: 832 x 480 (`length=124` for a 5 s clip).
# - Steps: 20, sampler `res_multistep`, scheduler `simple`.
# - The official template's 832 x 480 @ `length=362` (15 s) takes ~23 min on a 3090. The L4 will
#   be slower; budget ~50-70 min for 15 s on L4. For a smoke test, use `length=124`.


In [ ]:
#@title STEP 1 — Install ComfyUI v0.30.1+ and dependencies (Drive-persistent)

import os, sys, subprocess, time
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
COMFY_DIR = DRIVE_ROOT / 'ComfyUI_H3'
HF_CACHE = DRIVE_ROOT / 'AEI_3D_Cache' / 'H3_ComfyUI'

os.environ.setdefault('HF_HOME', str(HF_CACHE))
os.environ.setdefault('HUGGINGFACE_HUB_CACHE', str(HF_CACHE))
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.8')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

if not COMFY_DIR.exists():
    print(f'  Cloning ComfyUI to {COMFY_DIR} ...')
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY_DIR)], check=True)
else:
    print(f'  Reusing existing {COMFY_DIR}')

COMFY_DIR.mkdir(parents=True, exist_ok=True)

# Install pytorch cu130 (current Colab default). Force reinstall to override anything
# STEP 1 of other notebooks may have set.
print('  Installing pytorch cu130 ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'torch', 'torchvision', 'torchaudio',
                '--index-url', 'https://download.pytorch.org/whl/cu130'], check=False)

print('  Installing ComfyUI requirements ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '-r', str(COMFY_DIR / 'requirements.txt')], check=False)

import torch
print(f'  torch          : {torch.__version__}  (CUDA {torch.version.cuda})')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'  GPU            : {p.name}  ({p.total_memory / 1024**3:.1f} GB)')
    print(f'  Compute        : {p.major}.{p.minor}')
else:
    raise SystemExit('No GPU detected - ComfyUI needs CUDA.')


In [ ]:
#@title STEP 2 - Download Comfy-Org/MiniMax-H3 weights to Drive cache

import os, time
from pathlib import Path
from huggingface_hub import snapshot_download

# Comfy-Org/MiniMax-H3 layout.
#   diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors  (19.5 GB)
#   text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors          (15.7 GB)
#   vae/minimax_h3_video_vae_fp16.safetensors                            ( 5.0 GB)
#   vae/minimax_h3_audio_vae_fp32.safetensors                            ( 0.6 GB)
# Total: ~40.8 GB on disk. The Drive cache keeps them across Colab sessions.
# We symlink into ComfyUI's models/ subdirs so the loader finds them in-place.

WEIGHTS_DIR = Path('/content/drive/MyDrive/AEI_3D_Cache/H3_ComfyUI/weights')
COMFY_DIR = Path('/content/drive/MyDrive/ComfyUI_H3')

t0 = time.time()
weights = snapshot_download(
    repo_id='Comfy-Org/MiniMax-H3',
    local_dir=str(WEIGHTS_DIR),
    allow_patterns=[
        'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors',
        'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
        'vae/minimax_h3_video_vae_fp16.safetensors',
        'vae/minimax_h3_audio_vae_fp32.safetensors',
    ],
)
print(f'  Weights cached at {weights} in {time.time() - t0:.0f}s')

DIFF = COMFY_DIR / 'models' / 'diffusion_models'
TXT  = COMFY_DIR / 'models' / 'text_encoders'
VAE  = COMFY_DIR / 'models' / 'vae'
for d in (DIFF, TXT, VAE):
    d.mkdir(parents=True, exist_ok=True)

def _symlink(src, dst):
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    dst.symlink_to(src)

_symlink(WEIGHTS_DIR / 'diffusion_models' / 'minimax_h3_fl2va_pruned_int8_convrot.safetensors',
         DIFF / 'minimax_h3_fl2va_pruned_int8_convrot.safetensors')
_symlink(WEIGHTS_DIR / 'text_encoders' / 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
         TXT / 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors')
_symlink(WEIGHTS_DIR / 'vae' / 'minimax_h3_video_vae_fp16.safetensors',
         VAE / 'minimax_h3_video_vae_fp16.safetensors')
_symlink(WEIGHTS_DIR / 'vae' / 'minimax_h3_audio_vae_fp32.safetensors',
         VAE / 'minimax_h3_audio_vae_fp32.safetensors')

print('  Symlinks wired into ComfyUI/models/{diffusion_models,text_encoders,vae}/')
for d in (DIFF, TXT, VAE):
    for p in sorted(d.iterdir()):
        if p.is_symlink() or p.suffix == '.safetensors':
            print(f'    {d.name}/{p.name}')


In [ ]:
#@title STEP 3 - Launch ComfyUI subprocess (--disable-pinned-memory)

import os, sys, time, signal, subprocess, urllib.request, urllib.error, json
from pathlib import Path

COMFY_DIR = Path('/content/drive/MyDrive/ComfyUI_H3')
COMFY_HOST = '127.0.0.1'
COMFY_PORT = 8188
COMFY_URL  = f'http://{COMFY_HOST}:{COMFY_PORT}'

# Pinned memory is the entire reason this notebook exists: the default ComfyUI policy pins
# ~90% of host RAM. With 53 GB of RAM the pinned ceiling is ~47 GB, and the diffusers runtime
# OOM-kills long before the model fits. --disable-pinned-memory is the single flag that makes
# this recipe work on 24-32 GB RAM (see tonyd2wild/minimax-h3-local).
# --fp16-intermediates halves inter-node tensors and is cheap insurance.
LAUNCH_CMD = [
    sys.executable, 'main.py',
    '--listen', COMFY_HOST,
    '--port', str(COMFY_PORT),
    '--disable-pinned-memory',
    '--fp16-intermediates',
    '--disable-api-nodes',
    '--output-directory', str(COMFY_DIR / 'output'),
    '--input-directory', str(COMFY_DIR / 'input'),
]
env = os.environ.copy()

subprocess.run(['pkill', '-9', '-f', 'ComfyUI_H3.*main.py'], check=False)
time.sleep(2)

(COMFY_DIR / 'output').mkdir(parents=True, exist_ok=True)
(COMFY_DIR / 'input').mkdir(parents=True, exist_ok=True)

print(f'  Launching: {" ".join(LAUNCH_CMD)}')
log_path = COMFY_DIR / 'comfyui.log'
log_f = open(log_path, 'wb')
proc = subprocess.Popen(
    LAUNCH_CMD,
    cwd=str(COMFY_DIR),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    env=env,
    preexec_fn=os.setsid,
)
print(f'  PID: {proc.pid}, log: {log_path}')

ready = False
deadline = time.time() + 600
while time.time() < deadline:
    try:
        with urllib.request.urlopen(COMFY_URL + '/system_stats', timeout=2) as r:
            r.read()
        ready = True
        break
    except (urllib.error.URLError, ConnectionResetError, OSError):
        if proc.poll() is not None:
            print(f'  ComfyUI exited early with code {proc.returncode}. Last 40 log lines:')
            with open(log_path) as f:
                tail = f.read().splitlines()[-40:]
                for ln in tail:
                    print('   ', ln)
            raise SystemExit('ComfyUI failed to start.')
    time.sleep(2)

if not ready:
    proc.terminate()
    raise SystemExit(f'ComfyUI did not respond within 10 minutes. Tail of log:\n'
                     + '\n'.join(open(log_path).read().splitlines()[-20:]))

with urllib.request.urlopen(COMFY_URL + '/system_stats') as r:
    sysinfo = json.loads(r.read())
print(f'  ComfyUI ready: {sysinfo.get("system", {}).get("comfyui_version", "?")} on '
      f'{sysinfo.get("devices", [{}])[0].get("name", "?")}')

import builtins
builtins.H3_COMFY_PROC = proc
builtins.H3_COMFY_LOG = log_path
builtins.H3_COMFY_URL = COMFY_URL
builtins.H3_COMFY_DIR = COMFY_DIR
print('  Stored H3_COMFY_PROC / H3_COMFY_URL in builtins for STEP 4 access.')


In [ ]:
#@title STEP 4 - Gradio UI: t2v / fl2v with audio, runs through ComfyUI /prompt + /history

import os, json, time, uuid, shutil, urllib.request, urllib.parse, builtins, base64, io
from pathlib import Path

COMFY_URL  = getattr(builtins, 'H3_COMFY_URL', 'http://127.0.0.1:8188')
COMFY_DIR  = getattr(builtins, 'H3_COMFY_DIR', Path('/content/drive/MyDrive/ComfyUI_H3'))
PROC       = getattr(builtins, 'H3_COMFY_PROC', None)

import requests

def _poll_until_ready(url, timeout=10):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=2) as r:
                r.read()
            return True
        except Exception:
            time.sleep(1)
    return False

if not _poll_until_ready(COMFY_URL + '/system_stats'):
    raise SystemExit('ComfyUI is not running. Re-run STEP 3.')

def _build_workflow(mode, prompt, width, height, length, steps, seed,
                    unet_name, clip_name, video_vae, audio_vae, first_frame=None):
    p = {}
    p["6"]  = {"class_type": "UNETLoader",   "inputs": {"unet_name": unet_name, "weight_dtype": "default"}}
    p["13"] = {"class_type": "CLIPLoader",   "inputs": {"clip_name": clip_name, "type": "minimax", "device": "default"}}
    p["11"] = {"class_type": "VAELoader",    "inputs": {"vae_name": video_vae}}
    p["24"] = {"class_type": "VAELoader",    "inputs": {"vae_name": audio_vae}}
    if mode == 'fl2v' and first_frame is not None:
        p["30"] = {"class_type": "LoadImage", "inputs": {"image": first_frame}}
    inputs = {"clip": ["13", 0], "vae": ["11", 0],
              "width": width, "height": height, "length": length, "prompt": prompt}
    if mode == 'fl2v' and first_frame is not None:
        inputs["first_frame"] = ["30", 0]
    p["104"] = {"class_type": "MiniMaxH3ImageToVideo", "inputs": inputs}
    p["16"] = {"class_type": "BasicGuider",   "inputs": {"model": ["6", 0], "conditioning": ["104", 0]}}
    p["17"] = {"class_type": "KSamplerSelect","inputs": {"sampler_name": "res_multistep"}}
    p["9"]  = {"class_type": "BasicScheduler","inputs": {"model": ["6", 0], "scheduler": "simple",
                                                          "steps": steps, "denoise": 1}}
    p["15"] = {"class_type": "RandomNoise",   "inputs": {"noise_seed": seed}}
    p["14"] = {"class_type": "SamplerCustomAdvanced",
               "inputs": {"noise": ["15", 0], "guider": ["16", 0], "sampler": ["17", 0],
                          "sigmas": ["9", 0], "latent_image": ["104", 1]}}
    p["10"] = {"class_type": "VAEDecode",      "inputs": {"samples": ["14", 0], "vae": ["11", 0]}}
    p["23"] = {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["14", 0], "vae": ["24", 0]}}
    p["91"] = {"class_type": "CreateVideo",    "inputs": {"images": ["10", 0], "audio": ["23", 0], "fps": 24}}
    p["92"] = {"class_type": "SaveVideo",      "inputs": {"video": ["91", 0],
                                                           "filename_prefix": f"video/H3_{mode}",
                                                           "format": "auto", "codec": "auto"}}
    return {"prompt": p}


def _upload_image(local_path):
    with open(local_path, 'rb') as f:
        files = {'image': (Path(local_path).name, f, 'image/png')}
        data  = {'type': 'input', 'overwrite': 'true'}
        r = requests.post(COMFY_URL + '/upload/image', files=files, data=data, timeout=120)
    r.raise_for_status()
    return r.json()['name']


def _queue_workflow(wf, client_id):
    r = requests.post(COMFY_URL + '/prompt', json={"prompt": wf["prompt"], "client_id": client_id}, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f'ComfyUI rejected workflow: {r.status_code} {r.text[:500]}')
    return r.json()['prompt_id']


def _wait_for_history(prompt_id, timeout=3600, poll=3):
    deadline = time.time() + timeout
    while time.time() < deadline:
        r = requests.get(f'{COMFY_URL}/history/{prompt_id}', timeout=10)
        if r.status_code == 200 and prompt_id in r.json():
            return r.json()[prompt_id]
        time.sleep(poll)
    raise TimeoutError(f'Workflow {prompt_id} did not finish within {timeout}s')


def _download_outputs(history_entry, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for node_id, node_out in history_entry.get('outputs', {}).items():
        for kind in ('videos', 'images', 'audio'):
            for entry in node_out.get(kind, []):
                fn = entry.get('filename')
                if not fn:
                    continue
                subdir = entry.get('type', 'output')
                src_url = f'{COMFY_URL}/view?filename={urllib.parse.quote(fn)}&type={subdir}'
                local = out_dir / fn
                with urllib.request.urlopen(src_url) as r, open(local, 'wb') as f:
                    shutil.copyfileobj(r, f)
                paths.append(str(local))
    return paths


import gradio as gr

OUTPUT_DIR = COMFY_DIR / 'output'

def run_minimax_h3(mode, prompt, first_frame,
                    width, height, length, steps, seed,
                    unet_name, clip_name, video_vae, audio_vae,
                    progress=gr.Progress(track_tqdm=False)):
    if not prompt or not prompt.strip():
        raise gr.Error('MiniMax-H3 needs a non-empty prompt.')
    while length % 17 != 5:
        length += 1

    client_id = str(uuid.uuid4())
    first_frame_name = None
    if mode == 'fl2v' and first_frame is not None:
        first_frame_name = _upload_image(first_frame)

    wf = _build_workflow(mode, prompt, int(width), int(height), int(length), int(steps), int(seed),
                          unet_name, clip_name, video_vae, audio_vae,
                          first_frame=first_frame_name)
    progress(0.05, desc=f'Queueing {mode} workflow ({width}x{height}, {length} frames, {steps} steps)...')
    prompt_id = _queue_workflow(wf, client_id)
    progress(0.1, desc=f'Queued as {prompt_id[:8]}. Polling ComfyUI queue...')

    started = time.time()
    while True:
        h = _wait_for_history(prompt_id, timeout=120)
        if h.get('status', {}).get('completed'):
            break
        if h.get('status', {}).get('error'):
            raise gr.Error('ComfyUI failed: ' + json.dumps(h['status'].get('messages', []), indent=2)[:2000])
        elapsed = time.time() - started
        progress(min(0.1 + 0.85 * (elapsed / 600), 0.95),
                 desc=f'Denoising in progress... {elapsed:.0f}s elapsed ({(elapsed / max(int(steps),1)):.1f}s/step est.)')

    paths = _download_outputs(h, OUTPUT_DIR)
    progress(1.0, desc=f'Done in {time.time() - started:.0f}s. Outputs: {[Path(p).name for p in paths]}')
    if not paths:
        raise gr.Error('Workflow finished but no outputs were produced.')
    video_path = next((p for p in paths if p.endswith(('.mp4', '.webm', '.mov'))), paths[0])
    report = (f'`{width}x{height}`, {length} frames ({length/24:.3f}s), {steps} steps - '
              f'time {time.time() - started:.0f}s - seed {seed} - mode `{mode}`')
    return video_path, report


CANVASES = [
    ('832 x 480 - 16:9',         832, 480),
    ('768 x 432 - 16:9 fast',    768, 432),
    ('640 x 360 - 16:9 cheapest', 640, 360),
    ('480 x 832 - 9:16',         480, 832),
    ('432 x 768 - 9:16 fast',    432, 768),
]
DEFAULT_CANVAS = '832 x 480 - 16:9'
DEFAULT_LENGTH = 124  # 5 s at 24 fps

UNET = 'minimax_h3_fl2va_pruned_int8_convrot.safetensors'
CLIP = 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'
VVAE = 'minimax_h3_video_vae_fp16.safetensors'
AVAE = 'minimax_h3_audio_vae_fp32.safetensors'

with gr.Blocks(title='MiniMax-H3 (ComfyUI)') as demo:
    gr.Markdown('# MiniMax-H3 - joint video + audio\n\nBackend: ComfyUI subprocess (--disable-pinned-memory). Weights: Comfy-Org/MiniMax-H3 int8_convrot + NVFP4.')
    gr.Markdown('### Welcome\n\nMiniMax-H3 generates 5-15 s of synchronized video + stereo audio from a prompt or first frame. The model is hosted by a ComfyUI subprocess running on 127.0.0.1:8188 (started in STEP 3).\n\n**First run:** keep the defaults (832 x 480, 124 frames = 5 s, 20 steps).')
    canvas_state = gr.State(DEFAULT_CANVAS)
    canvas_dims = gr.State((832, 480))
    with gr.Row():
        with gr.Column():
            mode = gr.Radio(choices=[('Text-to-video (t2va)', 't2va'),
                                      ('Image-to-video (fl2va)', 'fl2va')],
                            value='t2va', label='Mode',
                            info='t2va is the most reliable; fl2va needs an explicit first frame.')
            prompt = gr.Textbox(lines=6,
                                value=('Cinematic medium shot of a red fox trotting through a snowy pine forest at dawn, '
                                       'snow crunching underfoot, natural handheld micro-movement, shallow depth of field, '
                                       '35mm film grain.\n\noverall_soundscape: forest ambience, distant birdsong, '
                                       'footfalls on frozen snow.\nnon_diegetic_music: none.'),
                                label='Prompt',
                                info='Supports [Shot N] scene direction and Dialogue: lines. Inline dialogue is spoken by the audio decoder.')
            first_frame = gr.Image(label='First frame (fl2va only)', type='filepath', sources=['upload'])
            canvas_dd = gr.Dropdown(choices=[c[0] for c in CANVASES], value=DEFAULT_CANVAS, label='Canvas',
                                    info='Smaller canvas + shorter length = dramatically faster.')
            length = gr.Slider(56, 362, value=DEFAULT_LENGTH, step=1, label='Length (frames at 24 fps)',
                               info='Snapped to the 17n+5 grid. 124 ~ 5 s, 362 ~ 15 s.')
            with gr.Row():
                steps = gr.Slider(10, 40, value=20, step=1, label='Steps',
                                  info='20 is the upstream default; 24-28 marginally improves quality.')
                seed  = gr.Number(value=42, precision=0, label='Seed',
                                  info='0 = random.')
            run = gr.Button('Generate', variant='primary')
        with gr.Column():
            video_out = gr.Video(label='Video + soundtrack')
            report    = gr.Markdown()

    def _set_dims(label):
        for n, w, h in CANVASES:
            if n == label:
                return (w, h)
        return (832, 480)
    canvas_dd.change(_set_dims, canvas_dd, canvas_dims)

    run.click(
        run_minimax_h3,
        [mode, prompt, first_frame,
         gr.State(832), gr.State(480), length, steps, seed,
         gr.State(UNET), gr.State(CLIP), gr.State(VVAE), gr.State(AVAE)],
        [video_out, report],
    )

    def _welcome():
        return (
            'ComfyUI is running. Submit a workflow via the Gradio UI on '
            'http://127.0.0.1:7860 or POST to /prompt with an API-format workflow JSON.',
        )
    demo.load(_welcome, None, [report])

demo.queue(default_concurrency_limit=1).launch(share=False, inline=False, prevent_thread_lock=True, server_port=7860)
import builtins
builtins.H3_DEMO = demo
from IPython.display import clear_output as _clear
_clear()
print('\nGradio UI: http://127.0.0.1:7860 (open the Colab proxy URL above)')


In [ ]:
#@title STEP 5 - Keep-alive + session summary (ComfyUI status)

import os, time, json, urllib.request, builtins

URL = getattr(builtins, 'H3_COMFY_URL', 'http://127.0.0.1:8188')

import IPython.display
display(IPython.display.Javascript("""
function KeepAlive() { console.log('Colab session kept alive at ' + new Date().toISOString()); }
setInterval(KeepAlive, 60000);
"""))

print('=' * 72)
print('MiniMax-H3 / ComfyUI session summary')
print('=' * 72)

try:
    with urllib.request.urlopen(URL + '/system_stats', timeout=5) as r:
        stats = json.loads(r.read())
    devs = stats.get('devices', [])
    for d in devs:
        free  = d.get('vram_free', 0) / 1024**3
        total = d.get('vram_total', 0) / 1024**3
        print(f'  GPU {d.get("name"):<24} {free:5.1f} GB free / {total:5.1f} GB total')
    print(f'  ComfyUI version  : {stats.get("system", {}).get("comfyui_version", "?")}')
    print(f'  Python version   : {stats.get("system", {}).get("python_version", "?")}')
    print(f'  Embedded at      : {URL}')
    print(f'  Log file         : {getattr(builtins, "H3_COMFY_LOG", "?")}')
    print(f'  Output dir       : {getattr(builtins, "H3_COMFY_DIR", "?")}/output')
    print()
    print('  ComfyUI subprocess is running. Submit a workflow via the Gradio UI on')
    print('  http://127.0.0.1:7860 or POST to /prompt with an API-format workflow JSON.')
except Exception as e:
    print(f'  [WARN] ComfyUI not reachable: {e}')


In [ ]:
#@title STEP 6 - Quick test (single video generation)

import time, json, uuid, urllib.request, requests
from pathlib import Path
import builtins

URL = getattr(builtins, 'H3_COMFY_URL', 'http://127.0.0.1:8188')
OUT = Path(getattr(builtins, 'H3_COMFY_DIR', Path('/content/drive/MyDrive/ComfyUI_H3'))) / 'output'

PROMPT = ('Cinematic medium shot of a red fox trotting through a snowy pine forest at dawn, '
          'snow crunching underfoot, natural handheld micro-movement, shallow depth of field, '
          '35mm film grain.\n\noverall_soundscape: forest ambience, distant birdsong, '
          'footfalls on frozen snow.\nnon_diegetic_music: none.')
WIDTH, HEIGHT, LENGTH, STEPS, SEED = 832, 480, 124, 20, 42

UNET = 'minimax_h3_fl2va_pruned_int8_convrot.safetensors'
CLIP = 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'
VVAE = 'minimax_h3_video_vae_fp16.safetensors'
AVAE = 'minimax_h3_audio_vae_fp32.safetensors'

while LENGTH % 17 != 5:
    LENGTH += 1

print(f'  Prompt    : {PROMPT[:60]}...')
print(f'  Canvas    : {WIDTH} x {HEIGHT}  ({LENGTH} frames = {LENGTH / 24:.3f}s at 24 fps)')
print(f'  Steps     : {STEPS}')
print(f'  Seed      : {SEED}')

wf = {"prompt": {
    "6":  {"class_type": "UNETLoader",   "inputs": {"unet_name": UNET, "weight_dtype": "default"}},
    "13": {"class_type": "CLIPLoader",   "inputs": {"clip_name": CLIP, "type": "minimax", "device": "default"}},
    "11": {"class_type": "VAELoader",    "inputs": {"vae_name": VVAE}},
    "24": {"class_type": "VAELoader",    "inputs": {"vae_name": AVAE}},
    "104":{"class_type": "MiniMaxH3ImageToVideo",
           "inputs": {"clip": ["13", 0], "vae": ["11", 0], "width": WIDTH, "height": HEIGHT,
                      "length": LENGTH, "prompt": PROMPT}},
    "16": {"class_type": "BasicGuider",   "inputs": {"model": ["6", 0], "conditioning": ["104", 0]}},
    "17": {"class_type": "KSamplerSelect","inputs": {"sampler_name": "res_multistep"}},
    "9":  {"class_type": "BasicScheduler","inputs": {"model": ["6", 0], "scheduler": "simple",
                                                       "steps": STEPS, "denoise": 1}},
    "15": {"class_type": "RandomNoise",   "inputs": {"noise_seed": SEED}},
    "14": {"class_type": "SamplerCustomAdvanced",
           "inputs": {"noise": ["15", 0], "guider": ["16", 0], "sampler": ["17", 0],
                      "sigmas": ["9", 0], "latent_image": ["104", 1]}},
    "10": {"class_type": "VAEDecode",      "inputs": {"samples": ["14", 0], "vae": ["11", 0]}},
    "23": {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["14", 0], "vae": ["24", 0]}},
    "91": {"class_type": "CreateVideo",    "inputs": {"images": ["10", 0], "audio": ["23", 0], "fps": 24}},
    "92": {"class_type": "SaveVideo",      "inputs": {"video": ["91", 0],
                                                        "filename_prefix": "video/H3_quicktest",
                                                        "format": "auto", "codec": "auto"}},
}}

client_id = str(uuid.uuid4())
t0 = time.time()
r = requests.post(URL + '/prompt', json={"prompt": wf["prompt"], "client_id": client_id}, timeout=60)
if r.status_code != 200:
    raise SystemExit(f'ComfyUI rejected the workflow: {r.status_code} {r.text[:1000]}')
prompt_id = r.json()['prompt_id']
print(f'\n  Queued: {prompt_id[:8]} ... polling /history')

last_report = 0
while True:
    h = requests.get(f'{URL}/history/{prompt_id}', timeout=10).json()
    if prompt_id in h:
        entry = h[prompt_id]
        if entry.get('status', {}).get('completed'):
            elapsed = time.time() - t0
            print(f'  Done in {elapsed:.0f}s ({(elapsed / STEPS):.1f}s/step).')
            break
        if entry.get('status', {}).get('error'):
            raise SystemExit('ComfyUI failed: ' + json.dumps(entry['status'].get('messages', []), indent=2)[:2000])
    elapsed = time.time() - t0
    if time.time() - last_report > 30:
        last_report = time.time()
        print(f'    ... {elapsed:.0f}s elapsed')
    time.sleep(3)

import shutil
paths = []
for node_id, node_out in entry.get('outputs', {}).items():
    for kind in ('videos', 'images', 'audio'):
        for out in node_out.get(kind, []):
            fn = out.get('filename')
            if not fn:
                continue
            subdir = out.get('type', 'output')
            url = f'{URL}/view?filename={urllib.parse.quote(fn)}&type={subdir}'
            local = OUT / fn
            with urllib.request.urlopen(url) as r, open(local, 'wb') as f:
                shutil.copyfileobj(r, f)
            paths.append(local)

print(f'\n  Outputs ({len(paths)}):')
for p in paths:
    print(f'    {p}')
video_path = next((str(p) for p in paths if str(p).endswith(('.mp4', '.webm', '.mov'))), str(paths[0]))
print(f'\n  Open: {video_path}')
from IPython.display import FileLink
display(FileLink(video_path))


In [ ]:
#@title STEP 7 - Batch generation from a JSON scene list

import json, time, uuid, urllib.request, requests, shutil, gc
from pathlib import Path
import builtins

URL = getattr(builtins, 'H3_COMFY_URL', 'http://127.0.0.1:8188')
OUT = Path(getattr(builtins, 'H3_COMFY_DIR', Path('/content/drive/MyDrive/ComfyUI_H3'))) / 'output'

UNET = 'minimax_h3_fl2va_pruned_int8_convrot.safetensors'
CLIP = 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'
VVAE = 'minimax_h3_video_vae_fp16.safetensors'
AVAE = 'minimax_h3_audio_vae_fp32.safetensors'

SCENES = [
  {"prompt": "A red fox in a snowy pine forest at dawn, slow dolly push-in, snow crunching underfoot.",
   "canvas": "832 x 480 - 16:9", "length": 124, "steps": 20, "seed": 42},
  {"prompt": "A busy night market, neon signs reflecting in puddles, sizzling street food, ambient chatter.",
   "canvas": "832 x 480 - 16:9", "length": 124, "steps": 20, "seed": 7},
  {"prompt": "A cellist playing a slow melody in an empty concert hall, warm stage lighting, distant applause at the end.",
   "canvas": "832 x 480 - 16:9", "length": 124, "steps": 20, "seed": 99}
]  #@param {type:"raw"}

CANVASES = {
    '832 x 480 - 16:9':          (832, 480),
    '768 x 432 - 16:9 fast':     (768, 432),
    '640 x 360 - 16:9 cheapest': (640, 360),
    '480 x 832 - 9:16':          (480, 832),
    '432 x 768 - 9:16 fast':     (432, 768),
}

if not isinstance(SCENES, list):
    raise SystemExit(f'Expected a JSON list, got {type(SCENES).__name__}')
print(f'  Loaded {len(SCENES)} scene(s)')

results = []
total_start = time.time()
for i, sc in enumerate(SCENES):
    prompt = sc.get('prompt', '').strip()
    if not prompt:
        print(f'  [{i}] SKIP: empty prompt')
        continue
    canvas_label = sc.get('canvas', '832 x 480 - 16:9')
    w, h = CANVASES.get(canvas_label, (832, 480))
    length = int(sc.get('length', 124))
    while length % 17 != 5:
        length += 1
    steps = int(sc.get('steps', 20))
    seed = int(sc.get('seed', 42))

    print(f'\n  [{i+1}/{len(SCENES)}] {w}x{h} {length}f {steps}steps seed={seed}')
    print(f'    {prompt[:80]}')

    wf = {"prompt": {
        "6":  {"class_type": "UNETLoader",   "inputs": {"unet_name": UNET, "weight_dtype": "default"}},
        "13": {"class_type": "CLIPLoader",   "inputs": {"clip_name": CLIP, "type": "minimax", "device": "default"}},
        "11": {"class_type": "VAELoader",    "inputs": {"vae_name": VVAE}},
        "24": {"class_type": "VAELoader",    "inputs": {"vae_name": AVAE}},
        "104":{"class_type": "MiniMaxH3ImageToVideo",
               "inputs": {"clip": ["13", 0], "vae": ["11", 0], "width": w, "height": h,
                          "length": length, "prompt": prompt}},
        "16": {"class_type": "BasicGuider",   "inputs": {"model": ["6", 0], "conditioning": ["104", 0]}},
        "17": {"class_type": "KSamplerSelect","inputs": {"sampler_name": "res_multistep"}},
        "9":  {"class_type": "BasicScheduler","inputs": {"model": ["6", 0], "scheduler": "simple",
                                                           "steps": steps, "denoise": 1}},
        "15": {"class_type": "RandomNoise",   "inputs": {"noise_seed": seed}},
        "14": {"class_type": "SamplerCustomAdvanced",
               "inputs": {"noise": ["15", 0], "guider": ["16", 0], "sampler": ["17", 0],
                          "sigmas": ["9", 0], "latent_image": ["104", 1]}},
        "10": {"class_type": "VAEDecode",      "inputs": {"samples": ["14", 0], "vae": ["11", 0]}},
        "23": {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["14", 0], "vae": ["24", 0]}},
        "91": {"class_type": "CreateVideo",    "inputs": {"images": ["10", 0], "audio": ["23", 0], "fps": 24}},
        "92": {"class_type": "SaveVideo",      "inputs": {"video": ["91", 0],
                                                            "filename_prefix": f"video/H3_batch_{i:03d}",
                                                            "format": "auto", "codec": "auto"}},
    }}

    t0 = time.time()
    r = requests.post(URL + '/prompt', json={"prompt": wf["prompt"], "client_id": str(uuid.uuid4())}, timeout=60)
    if r.status_code != 200:
        print(f'    FAIL: {r.status_code} {r.text[:300]}')
        continue
    pid = r.json()['prompt_id']
    while True:
        h = requests.get(f'{URL}/history/{pid}', timeout=10).json()
        if pid in h:
            entry = h[pid]
            if entry.get('status', {}).get('completed'):
                break
            if entry.get('status', {}).get('error'):
                print(f'    FAIL: {json.dumps(entry["status"].get("messages", []))[:300]}')
                continue
        time.sleep(5)
    elapsed = time.time() - t0
    for node_id, node_out in entry.get('outputs', {}).items():
        for v in node_out.get('videos', []):
            fn = v['filename']
            url = f'{URL}/view?filename={urllib.parse.quote(fn)}&type=output'
            local = OUT / fn
            with urllib.request.urlopen(url) as r2, open(local, 'wb') as f:
                shutil.copyfileobj(r2, f)
            results.append(str(local))
            print(f'    {local.name}  ({elapsed:.0f}s)')
    gc.collect()

total = time.time() - total_start
print(f'\nBatch complete: {len(results)}/{len(SCENES)} clips, total {total:.0f}s ({total / max(len(results),1):.0f}s/clip)')
for p in results:
    print(f'  {p}')
